# Pré-processamento e extração de features (ds004504)

Todo o processamento está em `ds004504_features.py`, o mesmo módulo usado no BrainLat.
Este notebook só organiza a execução: seleção da amostra, conferência de um sujeito,
lote com cache, consolidação e controle de qualidade.

As mudanças em relação à versão anterior estão em `CORRECOES.md`.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

import feature_sets as FS
import ds004504_features as F
import modeling as M

DATA_ROOT = Path("88 subjects")
CACHE_DIR = Path("features_ds004504") / FS.feature_tag()
OUT_CSV   = Path("eeg_features_ds004504.csv")
QC_CSV    = Path("qc_ds004504.csv")

print("features:", FS.feature_tag(), "| referência-alvo:", FS.TARGET_REFERENCE)
print("bandas:", FS.BANDS)
print(f"filtro {F.FILTER_BAND} Hz | ICA ajustada em {F.ICA_FIT_BAND} Hz | ASR cutoff {F.ASR_CUTOFF}")
print(f"épocas de {F.EPOCH_DURATION} s com {F.EPOCH_OVERLAP} s de sobreposição")
print(f"PAC: idpac={F.PAC_IDPAC}, {F.PAC_N_PERM} surrogates, até {F.PAC_MAX_SECONDS:.0f} s por sujeito")

features: v2.1-average | referência-alvo: average
bandas: {'Delta': (1.0, 4.0), 'Theta': (4.0, 8.0), 'Alpha': (8.0, 13.0), 'Beta': (13.0, 25.0), 'Gamma': (25.0, 45.0)}
filtro (0.5, 45.0) Hz | ICA ajustada em (1.0, 100.0) Hz | ASR cutoff 17
épocas de 4.0 s com 2.0 s de sobreposição
PAC: idpac=(2, 2, 4), 100 surrogates, até 300 s por sujeito


c:\Users\isabe\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Amostra

Seleção pelo rótulo do `participants.tsv` (grupos A e C), não pela posição do arquivo.

In [2]:
participants = pd.read_csv(DATA_ROOT / "participants.tsv", sep="\t")
participants = participants.rename(columns={"participant_id": "Subject_ID"})
amostra = participants[participants["Group"].isin([FS.POSITIVE_GROUP, FS.NEGATIVE_GROUP])].copy()
amostra["path"] = [DATA_ROOT / s / "eeg" / f"{s}_task-eyesclosed_eeg.set" for s in amostra["Subject_ID"]]
amostra["existe"] = amostra["path"].map(Path.exists)

print(amostra.groupby("Group").size().to_string())
print("arquivos ausentes:", amostra.loc[~amostra["existe"], "Subject_ID"].tolist() or "nenhum")
amostra = amostra[amostra["existe"]]

Group
A    36
C    29
arquivos ausentes: nenhum


## 2. Um sujeito do início ao fim (conferência)

In [4]:
r0 = amostra.iloc[0]
raw, ref = F.prepare_ds004504_raw(r0["path"])
print(f"{r0['Subject_ID']}: {len(raw.ch_names)} canais @ {raw.info['sfreq']} Hz, {raw.times[-1]:.0f} s")
print("referência:", {k: v for k, v in ref.items() if k != "detected_before"})
print("antes da troca:", ref["detected_before"])

cleaned, rep = F.clean_like_training(raw)
print(json.dumps({k: v for k, v in rep.items() if k != "ic_probabilities"}, indent=1, ensure_ascii=False))

trecho = cleaned.copy().crop(tmax=60.0)          # 60 s para a conferência ser rápida
df0 = pd.DataFrame(F.extract_epoch_features(trecho, r0["Subject_ID"]))
print(F.epoch_qc(df0))
df0[FS.ALL_FEATURES].describe().T[["mean", "min", "max"]]

sub-001: 19 canais @ 500.0 Hz, 600 s
referência: {'target': 'average', 'source': 'linked_ears', 'ref_channels': [], 'n_channels_after': 19}
antes da troca: {'n_eeg': 19, 'avg_ratio': 1.1371, 'looks_average': False, 'ear_or_mastoid_channels': [], 'custom_ref_applied': False}
{
 "n_ica_components": 18,
 "ic_labels": [
  "eye blink",
  "brain",
  "eye blink",
  "brain",
  "line noise",
  "eye blink",
  "line noise",
  "line noise",
  "line noise",
  "brain",
  "line noise",
  "line noise",
  "muscle artifact",
  "line noise",
  "line noise",
  "muscle artifact",
  "line noise",
  "line noise"
 ],
 "ic_excluded": [
  0,
  2,
  5,
  12,
  15
 ],
 "n_ic_excluded": 5,
 "iclabel_in_spec": true,
 "asr_fraction_samples_modified": 0.0,
 "flat_channels": [],
 "median_channel_std_uV": 4.462,
 "duration_s": 599.8
}


WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations


{'n_epochs': 29, 'rbp_sum_max_abs_dev': 2.220446049250313e-16, 'kuramoto_degenerate_epochs': 0, 'nan_features': []}


,mean,min,max
Delta_RBP,0.366645,0.253371,0.526287
Theta_RBP,0.335603,0.150972,0.518575
Alpha_RBP,0.094333,0.048881,0.136571
Beta_RBP,0.091119,0.065637,0.129562
Gamma_RBP,0.112300,0.071768,0.173505
Delta_Mean_Freq,2.013753,1.722146,2.282173
Theta_Mean_Freq,5.471327,4.979185,5.735285
Alpha_Mean_Freq,9.773140,9.475126,10.186173
Beta_Mean_Freq,18.313499,17.677560,19.066584
Gamma_Mean_Freq,32.278420,31.365921,33.314065


## 3. Lote com cache

Cada sujeito grava `<id>_epochs.csv` e `<id>_qc.json` em `features_ds004504/v<versão>/`.
Ao rodar de novo, quem já tem cache é pulado.

In [5]:
falhas = {}
for i, r in enumerate(amostra.itertuples(), 1):
    print(f"[{i}/{len(amostra)}] {r.Subject_ID}")
    try:
        F.process_ds004504_subject(r.Subject_ID, r.path, CACHE_DIR)
    except Exception as exc:
        falhas[r.Subject_ID] = f"{type(exc).__name__}: {exc}"
        print("  FALHOU:", falhas[r.Subject_ID])
print("\nfalhas:", falhas or "nenhuma")

[1/65] sub-001


WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations


  298 épocas, 5 ICs removidos, ASR alterou 0.0% das amostras, 278 s
[2/65] sub-002


WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations


  395 épocas, 6 ICs removidos, ASR alterou 0.0% das amostras, 343 s
[3/65] sub-003


...

  403 épocas, 6 ICs removidos, ASR alterou 0.0% das amostras, 346 s
[64/65] sub-064


WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations


  423 épocas, 4 ICs removidos, ASR alterou 0.0% das amostras, 371 s
[65/65] sub-065


WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations
WARNING | For inferences at p<0.05, it is recommended to perform at least n_perm=200 permutations


  441 épocas, 4 ICs removidos, ASR alterou 0.0% das amostras, 371 s

falhas: nenhuma


## 4. Consolidação

In [6]:
frames = [pd.read_csv(CACHE_DIR / f"{s}_epochs.csv") for s in amostra["Subject_ID"]
          if (CACHE_DIR / f"{s}_epochs.csv").exists()]
epochs = pd.concat(frames, ignore_index=True)
cov = [c for c in ("Group", "Age", "Gender", "MMSE") if c in amostra.columns]
epochs = epochs.merge(amostra[["Subject_ID"] + cov], on="Subject_ID", how="left", validate="many_to_one")
assert epochs["Group"].notna().all()

epochs.to_csv(OUT_CSV, index=False)
OUT_CSV.with_suffix(".meta.json").write_text(json.dumps({
    "feature_tag": FS.feature_tag(), "feature_version": FS.FEATURE_VERSION,
    "target_reference": FS.TARGET_REFERENCE, "bands": FS.BANDS,
    "n_subjects": int(epochs["Subject_ID"].nunique()), "n_epochs": int(len(epochs))},
    indent=2, ensure_ascii=False), encoding="utf-8")
qc = F.qc_table(CACHE_DIR)
qc.to_csv(QC_CSV, index=False)
print(f"{OUT_CSV}: {epochs['Subject_ID'].nunique()} sujeitos, {len(epochs)} épocas, {epochs.shape[1]} colunas")
print(f"{QC_CSV}: {len(qc)} sujeitos")

eeg_features_ds004504.csv: 65 sujeitos, 26823 épocas, 37 colunas
qc_ds004504.csv: 65 sujeitos


## 5. Controle de qualidade

Nada é removido automaticamente: os sinais abaixo indicam sujeitos a inspecionar.

In [7]:
print("referência-alvo:", qc["reference"].unique().tolist(), "| componentes da ICA:", qc["n_ica_components"].unique().tolist())
print("ICs removidos por sujeito:", qc["n_ic_excluded"].describe()[["min", "50%", "max"]].to_dict())
print("fração de amostras alteradas pelo ASR:", qc["asr_frac_modified"].describe()[["min", "50%", "max"]].round(3).to_dict())
print("sujeitos com canal plano:", qc.loc[qc["flat_channels"].astype(str).str.len() > 0, "Subject_ID"].tolist() or "nenhum")
print("maior desvio da soma da RBP em relação a 1:", f"{qc['rbp_sum_max_abs_dev'].max():.2e}")
print("épocas com Kuramoto degenerado (= 1/19):", int(qc["kuramoto_degenerate_epochs"].sum()))
print("sujeitos com feature ausente:", qc.loc[qc["nan_features"].astype(str).str.len() > 0, ["Subject_ID", "nan_features"]].values.tolist() or "nenhum")

subj = M.subject_table(epochs)
pac = subj[FS.PAC_COLS]
print("\ncorrelação entre as colunas de PAC (nível do sujeito):")
print(f"  Pearson : mín {pac.corr().where(~np.eye(6, dtype=bool)).min().min():.3f}")
print(f"  Spearman: mín {pac.corr(method='spearman').where(~np.eye(6, dtype=bool)).min().min():.3f}")
print("\nvalores extremos (escore z robusto > 5):")
print(M.outlier_report(subj).to_string(index=False) or "nenhum")

referência-alvo: ['average'] | componentes da ICA: [18, 16, 15, 12, 17, 14, 13]
ICs removidos por sujeito: {'min': 0.0, '50%': 3.0, 'max': 7.0}
fração de amostras alteradas pelo ASR: {'min': 0.0, '50%': 0.0, 'max': 1.0}
sujeitos com canal plano: nenhum
maior desvio da soma da RBP em relação a 1: 4.44e-16
épocas com Kuramoto degenerado (= 1/19): 0
sujeitos com feature ausente: nenhum

correlação entre as colunas de PAC (nível do sujeito):
  Pearson : mín -0.096
  Spearman: mín -0.075

valores extremos (escore z robusto > 5):
Subject_ID         feature    valor  mediana  z_robusto
   sub-021       Gamma_RBP 0.231826 0.058018        5.6
   sub-026  PAC_Delta_Beta 1.716528 0.085427        5.5
   sub-026 PAC_Delta_Gamma 1.780349 0.040908        5.2
   sub-048 PAC_Delta_Gamma 2.666932 0.040908        7.8
   sub-023  PAC_Theta_Beta 2.720460 0.017426        5.2
   sub-024  PAC_Theta_Beta 2.613069 0.017426        5.0
   sub-002  PAC_Alpha_Beta 4.877789 0.371115        5.8
   sub-025  PAC_Alpha_